<a href="https://colab.research.google.com/github/ethanresek/luminal-classifiers/blob/main/CS6140_RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
Clone github if working in colab
'''

import os
try:
    import google.colab
    os.system('git clone https://github.com/ethanresek/luminal-classifiers 2>/dev/null; cd /content/luminal-classifiers && git pull')
except ImportError:
    pass

In [11]:
import numpy as np
import pandas as pd
import sys, os
import joblib

from datetime import datetime
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score
print("Imports OK")

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Imports OK


In [4]:
'''
Split set up instructions for local vs colab running
'''
try:
    from google.colab import drive
    sys.path.append('/content/luminal-classifiers')
    from pre_process import preprocess, split_data

    # Change CSV path if necessary
    # '/content/luminal-classifiers/' should remain the start of the path
    CSV = '/content/luminal-classifiers/data/METABRIC_RNA_Mutation.csv'
    print('Working in Colab')
except ImportError:
    from pre_process import preprocess, split_data
    # Change CSV to path in local storage if needed
    CSV = 'data/METABRIC_RNA_Mutation.csv'
    print('Working locally')

Working locally


In [5]:
DF = pd.read_csv(CSV, low_memory=False)

# Specify which columns to keep from dataframe
Y_OLD_NAME = 'pam50_+_claudin-low_subtype'
KEEP = list(DF.columns[31:520]) + [Y_OLD_NAME]

# Split data and convert to float32 for Torch
X, y = preprocess(DF, KEEP)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_SEED)

In [6]:
# Set up RandomForestClassifier with balanced class weight to account for the imbalance in the class spread
rf = RandomForestClassifier(class_weight='balanced')

# Distribution of possible values for hyperparameters
p_dist = {
    'n_estimators': [100, 200, 500, 1000],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.1, 0.25, 0.5],
    'max_leaf_nodes': [None, 50, 100, 200, 500],
    'min_impurity_decrease': [0.0, 0.001, 0.005, 0.01, 0.05],
    'max_samples': [0.5, 0.7, 0.8, 0.9, None],
    'criterion': ['gini', 'entropy', 'log_loss']
}

{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 10, 'min_impurity_decrease': 0.005, 'max_samples': 0.9, 'max_leaf_nodes': 100, 'max_features': 0.1, 'max_depth': 10, 'criterion': 'log_loss'}


In [ ]:
# Run randomized search to find optimal hyperparameters
rand_search = RandomizedSearchCV(rf, param_distributions=p_dist, scoring='f1', n_iter=100, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), random_state=RANDOM_SEED)
rand_search.fit(X_train, y_train)
print(rand_search.best_params_)

# For time purposes, GridSearch was not run on the Base model

In [9]:
# Produce F1, Balanced Accuracy, and ROC AUC scores
final_rf = rand_search.best_estimator_

y_pred = final_rf.predict(X_test)

print('F1:', f1_score(y_test, y_pred))
print('Balanced Accuracy:', balanced_accuracy_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_pred))

F1: 0.9072164948453608
Balanced Accuracy: 0.8977464788732394
ROC AUC: 0.8977464788732394


In [16]:
# Store timestamped results of RandomizedSearch and input data sets
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
joblib.dump({
    'model': rand_search.best_estimator_,
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test
}, f'models/tuned_rf_{timestamp}.joblib')

['models/tuned_rf_20260411_165750.joblib']